# 3b — Model Sweeps (v1.0)

Unified sweep notebook for the Aq2/Kq2 project.  
Consolidates all tests from `3a_PARAM_SWEEPS`, `3a_FEATURE_SWEEP`, and `3c_SIMILARITY_SWEEP`.

| § | Stage | Description | Est. wall time |
|---|---|---|---|
| 2 | **QRF param sweep** | Monte Carlo search over hyperparameter grid | 2–4 h |
| 3 | **Feature ablation (LOO)** | Leave-one-out ΔR² | ~20 min |
| 4 | **MC feature subset sweep** | Random subset R² + per-feature stats | 3–5 h |
| 5 | **Greedy forward selection** | Iterative best-feature build | 2–4 h |
| 6 | **Sim sanity check** | Baseline σ=1, K=7 | <1 min |
| 7 | **Sim σ optimisation** | Optuna joint bandwidth optimisation | 30–60 min |
| 8 | **Sim K sweep** | Grid search over K values | ~5 min |
| 9 | **Sim feature sensitivity** | LOO, single-feature, σ-sensitivity | 10–20 min |

All expensive stages have `RUN_*` toggles — set `False` to reload saved CSVs.  
Results saved incrementally; safe to interrupt at any time.

**Prerequisites:** `1_Import.ipynb` has been run and `data/IHFC_obs.parquet` exists.


## 0. Toggles

In [ ]:
# ── QRF sweeps ──────────────────────────────────────────────────────────────
RUN_PARAM_SWEEP    = True   # False → load saved param_results.csv
RUN_ABLATION       = True   # False → load saved ablation_results.csv
RUN_MONTE_CARLO    = True   # False → load saved mc_results.csv
RUN_GREEDY_FORWARD = True   # False → load saved greedy_forward.csv

# ── Similarity sweeps ────────────────────────────────────────────────────────
RUN_SIGMA_SWEEP    = True   # False → load saved sigma_results.csv
RUN_K_SWEEP        = True   # False → load saved k_results.csv
RUN_FEATURE_SENS   = True   # master toggle for §9 (LOO / single / σ-sens)
RUN_LOO            = True
RUN_SINGLE         = True
RUN_SIGMA_SENS     = True

# ── Sweep resolution ─────────────────────────────────────────────────────────
K_FIXED_FOR_SIGMA  = 7.0    # K held fixed during σ optimisation
K_SWEEP_STEPS      = 40     # grid points in K sweep
SIGMA_SENS_STEPS   = 15     # σ grid points per feature in sensitivity sweep
MC_MIN_FEATS       = 10     # minimum feature subset size for MC sweep
MC_MAX_FEATS       = None   # None → len(obs_sweep) - 1
PARAM_RANDOM_SEED  = 2026
MC_RANDOM_SEED     = 2025
GREEDY_RANDOM_SEED = 2025


## 1. Imports & setup

In [ ]:
import sys, json, time, warnings, platform, psutil
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from pathlib import Path
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from quantile_forest import RandomForestQuantileRegressor

from config import (
    parquet_ref, obs_sweep, obs_model,
    q_clip_min, q_clip_max, random_state,
    output_root, fig_root, fig_dpi, sweep_dir,
    PARAM_N_RUNS, MC_N_RUNS, N_OPTUNA_TRIALS,
    PARAM_GRID, CV_FOLDS_SIM, SIGMA_BOUNDS, K_RANGE,
    HIST_BIN_WIDTH, SIM_BATCH_SIZE,
    QRF_N_ESTIMATORS, QRF_MAX_FEATURES, QRF_MIN_SAMPLES_LEAF,
    QRF_MAX_DEPTH, QRF_N_JOBS,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Output directories ────────────────────────────────────────────────────
sweep_dir.mkdir(parents=True, exist_ok=True)
fig_root.mkdir(parents=True, exist_ok=True)

print(f"sweep_dir : {sweep_dir}")
print(f"fig_root  : {fig_root}")


### 1b. Load reference data

In [ ]:
df_ref = pd.read_parquet(parquet_ref)

# Validate obs_sweep against parquet
available_sweep = [f for f in obs_sweep if f in df_ref.columns]
missing_sweep   = [f for f in obs_sweep if f not in df_ref.columns]
if missing_sweep:
    warnings.warn(
        f"\n⚠ obs_sweep columns MISSING from parquet:\n  {missing_sweep}"
        f"\n  Re-run 1_Import.ipynb or update recipe.py / obs_sweep in config.",
        stacklevel=2,
    )
    raise RuntimeError(f"Missing sweep features: {missing_sweep}")

# Working dataframe: drop rows with any NaN in sweep features or target
cols_needed = available_sweep + ["q"]
if "weight" in df_ref.columns:
    cols_needed.append("weight")

work = (df_ref[cols_needed]
        .replace([np.inf, -np.inf], np.nan)
        .dropna())
work = work[(work["q"] >= q_clip_min) & (work["q"] <= q_clip_max)]

y_phys  = work["q"].values
w_all   = work["weight"].values if "weight" in work.columns else np.ones(len(work))
X_all   = work[available_sweep].values

print(f"Reference rows    : {len(work):,}")
print(f"Sweep features    : {len(available_sweep)}")
print(f"Features          : {available_sweep}")


### 1c. Train / calibration split (shared across all QRF sweeps)

In [ ]:
CALIBRATION_FRAC = 0.20   # 80/20 split for QRF sweeps

(X_train, X_cal,
 y_train_phys, y_cal_phys,
 w_train, _) = train_test_split(
    X_all, y_phys, w_all,
    test_size=CALIBRATION_FRAC,
    random_state=random_state,
)
print(f"Train : {len(X_train):,}   Cal : {len(X_cal):,}")


---
## 2. QRF Parameter Sweep

Monte Carlo search over `PARAM_GRID`.  
Results saved to `output/sweeps/param_results.csv` every 5 runs.


In [ ]:
param_csv = sweep_dir / "param_results.csv"
rng_p = np.random.default_rng(PARAM_RANDOM_SEED)
_t_param_start = time.time()

# Resume if partial results exist
if param_csv.exists():
    param_rows = pd.read_csv(param_csv).to_dict("records")
    print(f"Resuming from {len(param_rows)} existing runs.")
else:
    param_rows = []

runs_needed = PARAM_N_RUNS - len(param_rows)

if RUN_PARAM_SWEEP and runs_needed > 0:
    print(f"Runs to do: {runs_needed}")
    for _ in range(runs_needed):
        params    = {k: rng_p.choice(v) for k, v in PARAM_GRID.items()}
        use_log   = bool(params["use_log_target"])
        clip_max  = float(params["q_clip_max"])
        cal_frac  = float(params["cal_frac"])

        mask   = y_phys <= clip_max
        X_run  = X_all[mask]
        y_run  = y_phys[mask]
        w_run  = w_all[mask]

        X_tr, X_ca, ytr_phys, yca_phys, wtr, _ = train_test_split(
            X_run, y_run, w_run,
            test_size=cal_frac, random_state=random_state,
        )
        y_tr = np.log(ytr_phys) if use_log else ytr_phys

        sc = StandardScaler()
        Xtr_s = sc.fit_transform(X_tr)
        Xca_s = sc.transform(X_ca)

        mf = params["max_features"]
        if mf not in ("sqrt", "log2"):
            mf = float(mf)
        md_val = None if params["max_depth"] is None else int(params["max_depth"])

        qrf = RandomForestQuantileRegressor(
            n_estimators    = int(params["n_estimators"]),
            max_depth       = md_val,
            min_samples_leaf= int(params["min_samples_leaf"]),
            max_features    = mf,
            random_state    = random_state,
            n_jobs          = QRF_N_JOBS,
        )
        t0 = time.time()
        qrf.fit(Xtr_s, y_tr, sample_weight=wtr)
        elapsed = time.time() - t0

        preds_raw  = qrf.predict(Xca_s, quantiles=[0.05, 0.50, 0.95])
        preds_phys = np.clip(
            np.exp(preds_raw) if use_log else preds_raw,
            q_clip_min, q_clip_max,
        )
        p05, p50, p95 = preds_phys.T

        row = {
            "run"              : len(param_rows) + 1,
            "n_estimators"     : int(params["n_estimators"]),
            "max_depth"        : str(params["max_depth"]),
            "min_samples_leaf" : int(params["min_samples_leaf"]),
            "max_features"     : str(params["max_features"]),
            "use_log_target"   : use_log,
            "q_clip_max"       : clip_max,
            "cal_frac"         : cal_frac,
            "n_train"          : len(X_tr),
            "n_cal"            : len(X_ca),
            "rmse"             : float(np.sqrt(mean_squared_error(yca_phys, p50))),
            "mae"              : float(mean_absolute_error(yca_phys, p50)),
            "r2"               : float(r2_score(yca_phys, p50)),
            "picp"             : float(np.mean((yca_phys >= p05) & (yca_phys <= p95))),
            "elapsed_s"        : elapsed,
        }
        param_rows.append(row)

        if len(param_rows) % 5 == 0:
            pd.DataFrame(param_rows).to_csv(param_csv, index=False)

        n = len(param_rows)
        print(f"{n:3d}/{PARAM_N_RUNS} log={use_log} depth={str(params['max_depth']):4s} "
              f"leaf={int(params['min_samples_leaf']):2d} mf={str(params['max_features']):4s} "
              f"clip={clip_max:.3f} R²={row['r2']:.4f} RMSE={row['rmse']*1000:.2f} t={elapsed:.0f}s")

    param_df = pd.DataFrame(param_rows)
    param_df.to_csv(param_csv, index=False)
    print(f"\nComplete. Saved {param_csv}")

elif not RUN_PARAM_SWEEP:
    if param_csv.exists():
        param_df = pd.read_csv(param_csv)
        print(f"Loaded {param_csv} ({len(param_df)} runs)")
    else:
        print("No saved results — set RUN_PARAM_SWEEP = True.")
        param_df = None
else:
    param_df = pd.read_csv(param_csv)
    print(f"Already at {PARAM_N_RUNS} runs. Loaded {param_csv}.")

_t_param_end = time.time()


### 2a. Parameter sensitivity figure

In [ ]:
if param_df is not None and len(param_df) > 0:
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    sweep_params = ["max_depth", "min_samples_leaf", "max_features", "n_estimators"]

    for ax, param in zip(axes, sweep_params):
        grouped = param_df.groupby(param)["r2"]
        labels  = [str(k) for k in grouped.groups.keys()]
        data    = [grouped.get_group(k).values for k in grouped.groups.keys()]
        ax.boxplot(data, labels=labels, vert=True)
        ax.set_title(param, fontsize=10)
        ax.set_ylabel("Cal R²", fontsize=9)
        ax.tick_params(axis="x", labelsize=8, rotation=45)
        ax.grid(True, axis="y", alpha=0.3)

    fig.suptitle(
        f"QRF param sweep | {len(param_df)} runs | "
        f"best R²={param_df['r2'].max():.4f}",
        fontsize=11, fontweight="bold",
    )
    fig.tight_layout()
    fig.savefig(sweep_dir / "param_sensitivity.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/param_sensitivity.png")


---
## 3. Feature ablation (leave-one-out)

Trains QRF with each feature dropped in turn. ΔR² relative to full model.


In [ ]:
def train_and_evaluate(feat_idx, label=""):
    """Train QRF on column subset feat_idx. Returns metrics dict (physical W/m²)."""
    Xtr = X_train[:, feat_idx]
    Xca = X_cal[:, feat_idx]

    sc = StandardScaler()
    Xtr_s = sc.fit_transform(Xtr)
    Xca_s = sc.transform(Xca)

    qrf = RandomForestQuantileRegressor(
        n_estimators     = QRF_N_ESTIMATORS,
        max_features     = QRF_MAX_FEATURES,
        max_depth        = QRF_MAX_DEPTH,
        min_samples_leaf = QRF_MIN_SAMPLES_LEAF,
        random_state     = random_state,
        n_jobs           = QRF_N_JOBS,
    )
    t0 = time.time()
    qrf.fit(Xtr_s, y_train_phys, sample_weight=w_train)
    elapsed = time.time() - t0

    preds_raw  = qrf.predict(Xca_s, quantiles=[0.05, 0.50, 0.95])
    preds_phys = np.clip(preds_raw, q_clip_min, q_clip_max)
    p05, p50, p95 = preds_phys.T

    return {
        "label"   : label,
        "n_feats" : len(feat_idx),
        "rmse"    : float(np.sqrt(mean_squared_error(y_cal_phys, p50))),
        "mae"     : float(mean_absolute_error(y_cal_phys, p50)),
        "r2"      : float(r2_score(y_cal_phys, p50)),
        "picp"    : float(np.mean((y_cal_phys >= p05) & (y_cal_phys <= p95))),
        "elapsed_s": elapsed,
    }

# ── Baseline (all sweep features) ────────────────────────────────────────
all_idx = list(range(len(available_sweep)))
print("Training baseline (all sweep features)…")
baseline = train_and_evaluate(all_idx, "BASELINE")
baseline_r2   = baseline["r2"]
baseline_rmse = baseline["rmse"]
print(f"  RMSE {baseline_rmse*1000:.2f} mW/m²  R² {baseline_r2:.4f}  "
      f"PICP {baseline['picp']:.4f}  t={baseline['elapsed_s']:.1f}s")
print(f"  (Est. LOO time: {len(available_sweep)*baseline['elapsed_s']/60:.0f} min)")


In [ ]:
ablation_csv = sweep_dir / "ablation_results.csv"
_t_abl_start = time.time()

if RUN_ABLATION:
    abl_rows = []
    n = len(available_sweep)
    for i, feat in enumerate(available_sweep):
        idx_keep = [j for j in range(n) if j != i]
        res = train_and_evaluate(idx_keep, f"drop_{feat}")
        delta = res["r2"] - baseline_r2
        tag = "HELPFUL" if delta < -0.002 else ("HARMFUL" if delta > 0.002 else "neutral")
        abl_rows.append({
            "feature" : feat,
            "r2_without": res["r2"],
            "delta_r2"  : delta,
            "verdict"   : tag,
        })
        pd.DataFrame(abl_rows).to_csv(ablation_csv, index=False)
        print(f"  [{i+1:2d}/{n}] drop {feat:<22s} R²={res['r2']:.4f} Δ={delta:+.4f} {tag}")

    abl_df = pd.DataFrame(abl_rows).sort_values("delta_r2", ascending=False)
    abl_df.to_csv(ablation_csv, index=False)
    print(f"\nSaved {ablation_csv}")

elif ablation_csv.exists():
    abl_df = pd.read_csv(ablation_csv)
    print(f"Loaded {ablation_csv}")
else:
    abl_df = None
    print("No saved results — set RUN_ABLATION = True.")

_t_abl_end = time.time()


### 3a. Ablation figure

In [ ]:
if abl_df is not None:
    df_plot = abl_df.sort_values("delta_r2")
    n_feat  = len(df_plot)
    y_pos   = np.arange(n_feat)
    colors  = ["#1a9850" if d < -0.002 else "#d73027" if d > 0.002 else "#999999"
               for d in df_plot["delta_r2"]]

    fig, ax = plt.subplots(figsize=(8, max(5, n_feat * 0.38)))
    ax.barh(y_pos, df_plot["delta_r2"], color=colors, alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df_plot["feature"], fontsize=8)
    ax.axvline(0, color="k", lw=0.8)
    ax.set_xlabel("ΔR² (drop feature vs full model)", fontsize=9)
    ax.set_title(
        f"Leave-one-out | baseline R²={baseline_r2:.4f} | "
        f"green=helpful, red=harmful, grey=neutral",
        fontsize=10,
    )
    ax.grid(True, axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(sweep_dir / "ablation_chart.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/ablation_chart.png")


---
## 4. Monte Carlo feature subset sweep

Random feature subsets → R² distribution. Reveals which features appear in
high-performing subsets most frequently.


In [ ]:
mc_csv      = sweep_dir / "mc_results.csv"
mc_stat_csv = sweep_dir / "mc_feature_stats.csv"
rng_mc      = np.random.default_rng(MC_RANDOM_SEED)
_t_mc_start = time.time()

n_feats_total = len(available_sweep)
mc_max        = (n_feats_total - 1) if MC_MAX_FEATS is None else MC_MAX_FEATS

if mc_csv.exists():
    mc_rows = pd.read_csv(mc_csv).to_dict("records")
    print(f"Resuming from {len(mc_rows)} existing MC runs.")
else:
    mc_rows = []

runs_needed = MC_N_RUNS - len(mc_rows)

if RUN_MONTE_CARLO and runs_needed > 0:
    print(f"MC runs to do: {runs_needed}")
    for _ in range(runs_needed):
        n_sub = int(rng_mc.integers(MC_MIN_FEATS, mc_max + 1))
        idx   = sorted(rng_mc.choice(n_feats_total, size=n_sub, replace=False).tolist())
        feats = [available_sweep[i] for i in idx]
        res   = train_and_evaluate(idx, f"mc_{len(mc_rows)+1}")

        row = {
            "run"     : len(mc_rows) + 1,
            "n_feats" : n_sub,
            "r2"      : res["r2"],
            "rmse"    : res["rmse"],
            "features": ",".join(feats),
        }
        mc_rows.append(row)

        if len(mc_rows) % 10 == 0:
            pd.DataFrame(mc_rows).to_csv(mc_csv, index=False)
            print(f"  {len(mc_rows)}/{MC_N_RUNS}  n_feats={n_sub}  R²={res['r2']:.4f}")

    mc_df = pd.DataFrame(mc_rows)
    mc_df.to_csv(mc_csv, index=False)
    print(f"\nSaved {mc_csv}")

elif not RUN_MONTE_CARLO:
    if mc_csv.exists():
        mc_df = pd.read_csv(mc_csv)
        print(f"Loaded {mc_csv} ({len(mc_df)} runs)")
    else:
        mc_df = None
        print("No saved results — set RUN_MONTE_CARLO = True.")
else:
    mc_df = pd.read_csv(mc_csv)
    print(f"Already at {MC_N_RUNS} runs.")

# Per-feature stats: fraction of runs where feature is present AND R² > median
if mc_df is not None and len(mc_df) > 0:
    r2_median  = mc_df["r2"].median()
    top_mask   = mc_df["r2"] >= r2_median
    feat_stats = []
    for feat in available_sweep:
        in_run     = mc_df["features"].str.contains(feat)
        inclusion  = float(in_run.mean())
        top_incl   = float(in_run[top_mask].mean()) if top_mask.sum() > 0 else 0.0
        feat_stats.append({"feature": feat, "inclusion_rate": inclusion,
                           "top_quartile_rate": top_incl})
    mc_stat_df = pd.DataFrame(feat_stats).sort_values("top_quartile_rate", ascending=False)
    mc_stat_df.to_csv(mc_stat_csv, index=False)
    print(f"Saved {mc_stat_csv}")

_t_mc_end = time.time()


### 4a. MC feature chart + R² vs subset size

In [ ]:
if mc_df is not None and len(mc_df) > 0 and 'mc_stat_df' in dir():
    fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(available_sweep) * 0.38)))

    # Left: per-feature inclusion rate in top-50% runs
    df_s  = mc_stat_df.sort_values("top_quartile_rate")
    y_pos = np.arange(len(df_s))
    axes[0].barh(y_pos, df_s["top_quartile_rate"], alpha=0.8, color="steelblue", label="top-50%")
    axes[0].barh(y_pos, df_s["inclusion_rate"], alpha=0.3, color="steelblue", label="all runs")
    axes[0].set_yticks(y_pos); axes[0].set_yticklabels(df_s["feature"], fontsize=8)
    axes[0].set_xlabel("Inclusion rate", fontsize=9)
    axes[0].set_title("MC feature inclusion (top-50% runs vs all runs)", fontsize=10)
    axes[0].legend(fontsize=8); axes[0].grid(True, axis="x", alpha=0.3)

    # Right: R² vs subset size scatter
    axes[1].scatter(mc_df["n_feats"], mc_df["r2"], alpha=0.3, s=8, color="steelblue")
    axes[1].axhline(baseline_r2, color="firebrick", lw=1.5, ls="--",
                    label=f"Baseline R²={baseline_r2:.4f}")
    axes[1].set_xlabel("Feature subset size", fontsize=9)
    axes[1].set_ylabel("Cal R²", fontsize=9)
    axes[1].set_title("R² vs subset size", fontsize=10)
    axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

    fig.suptitle(f"MC sweep | {len(mc_df)} runs", fontsize=11, fontweight="bold")
    fig.tight_layout()
    fig.savefig(sweep_dir / "mc_feature_chart.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/mc_feature_chart.png")

    # Separate R² vs size figure (recovered from 3a_FEATURE_SWEEP)
    fig2, ax2 = plt.subplots(figsize=(8, 5))
    ax2.scatter(mc_df["n_feats"], mc_df["r2"], alpha=0.3, s=8, color="steelblue")
    # Running median
    bin_edges  = np.arange(MC_MIN_FEATS - 0.5, mc_max + 1.5, 1)
    bin_mids   = 0.5 * (bin_edges[:-1] + bin_edges[1:])
    binned_med = [mc_df["r2"][
                      (mc_df["n_feats"] >= bin_edges[i]) &
                      (mc_df["n_feats"] <  bin_edges[i+1])
                  ].median() for i in range(len(bin_mids))]
    ax2.plot(bin_mids, binned_med, "r-", lw=1.5, label="running median")
    ax2.axhline(baseline_r2, color="firebrick", lw=1.5, ls="--",
                label=f"Baseline (all feats) R²={baseline_r2:.4f}")
    ax2.set_xlabel("Number of features in subset", fontsize=10)
    ax2.set_ylabel("Cal R²", fontsize=10)
    ax2.set_title("R² vs feature subset size", fontsize=11)
    ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(sweep_dir / "mc_r2_vs_size.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/mc_r2_vs_size.png")


---
## 5. Greedy forward selection

Iteratively adds the single feature that most improves R².


In [ ]:
greedy_csv  = sweep_dir / "greedy_forward.csv"
rng_gr      = np.random.default_rng(GREEDY_RANDOM_SEED)
_t_gr_start = time.time()

if RUN_GREEDY_FORWARD:
    remaining = list(range(len(available_sweep)))
    selected  = []
    greedy_rows = []

    while remaining:
        best_r2, best_idx = -np.inf, None
        for idx in remaining:
            trial_idx = selected + [idx]
            res = train_and_evaluate(trial_idx)
            if res["r2"] > best_r2:
                best_r2, best_idx = res["r2"], idx

        selected.append(best_idx)
        remaining.remove(best_idx)
        feat_name = available_sweep[best_idx]
        greedy_rows.append({
            "step"           : len(selected),
            "feature_added"  : feat_name,
            "n_feats"        : len(selected),
            "r2"             : best_r2,
            "features_so_far": ",".join(available_sweep[i] for i in selected),
        })
        pd.DataFrame(greedy_rows).to_csv(greedy_csv, index=False)
        print(f"  Step {len(selected):2d}: +{feat_name:<22s} R²={best_r2:.4f}")

    greedy_df = pd.DataFrame(greedy_rows)
    print(f"\nSaved {greedy_csv}")

elif greedy_csv.exists():
    greedy_df = pd.read_csv(greedy_csv)
    print(f"Loaded {greedy_csv}")
else:
    greedy_df = None
    print("No saved results — set RUN_GREEDY_FORWARD = True.")

_t_gr_end = time.time()


### 5a. Greedy forward selection curve

In [ ]:
if greedy_df is not None and len(greedy_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(greedy_df["n_feats"], greedy_df["r2"], "o-", ms=5,
            color="steelblue", lw=1.5)
    ax.axhline(baseline_r2, color="firebrick", lw=1.5, ls="--",
               label=f"Full model R²={baseline_r2:.4f}")
    for _, row in greedy_df.iterrows():
        ax.annotate(row["feature_added"],
                    xy=(row["n_feats"], row["r2"]),
                    xytext=(3, 3), textcoords="offset points",
                    fontsize=6, color="grey")
    ax.set_xlabel("Number of features", fontsize=10)
    ax.set_ylabel("Cal R²", fontsize=10)
    ax.set_title("Greedy forward selection", fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(sweep_dir / "greedy_curve.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/greedy_curve.png")


---
## 6. Write `qrf_params.json`

Saves best QRF hyperparameters and obs_model feature list for downstream notebooks.


In [ ]:
qrf_json = sweep_dir / "qrf_params.json"

if param_df is not None and len(param_df) > 0:
    best_row = param_df.loc[param_df["r2"].idxmax()]
    best_qrf = {
        "model"            : "QRF",
        "sweep_notebook"   : "3b_MODEL_SWEEPS",
        "best_r2_cal"      : float(best_row["r2"]),
        "n_estimators"     : int(best_row["n_estimators"]),
        "max_depth"        : None if str(best_row["max_depth"]) == "None"
                                  else int(best_row["max_depth"]),
        "min_samples_leaf" : int(best_row["min_samples_leaf"]),
        "max_features"     : (float(best_row["max_features"])
                               if best_row["max_features"] not in ("sqrt", "log2")
                               else best_row["max_features"]),
        "use_log_target"   : bool(best_row["use_log_target"]),
        "q_clip_max"       : float(best_row["q_clip_max"]),
        "cal_frac"         : float(best_row["cal_frac"]),
        "obs_model"        : obs_model,   # 22 curated features
        "n_sweep_runs"     : len(param_df),
    }
    with open(qrf_json, "w") as fp:
        json.dump(best_qrf, fp, indent=2)
    print(f"Saved {qrf_json}")
    print(f"Best R² cal : {best_qrf['best_r2_cal']:.4f}")
    print(f"n_estimators: {best_qrf['n_estimators']}")
    print(f"max_depth   : {best_qrf['max_depth']}")
    print(f"min_samples : {best_qrf['min_samples_leaf']}")
    print(f"max_features: {best_qrf['max_features']}")
else:
    print("⚠ No param_df — run §2 first or set RUN_PARAM_SWEEP = True.")


---
## 7. Similarity model — sanity check

Verify baseline CV R² (σ=1, K=7) before launching Optuna.
Uses `obs_sweep` (same as QRF sweeps) for comparability.


In [ ]:
# ── CV folds for similarity sweep (shared across §§7–9) ──────────────────
df_sim = (df_ref[[*available_sweep, "q"]]
          .replace([np.inf, -np.inf], np.nan)
          .dropna())
df_sim = df_sim[(df_sim["q"] >= q_clip_min) & (df_sim["q"] <= q_clip_max)]

X_sim  = df_sim[available_sweep].values   # raw, unstandardised
y_sim  = df_sim["q"].values

kf         = KFold(n_splits=CV_FOLDS_SIM, shuffle=True, random_state=random_state)
fold_splits = list(kf.split(X_sim))

SIGMA_START = {f: 1.0 for f in available_sweep}

print(f"Sim reference rows : {len(df_sim):,}")
print(f"Sim features       : {len(available_sweep)}")


In [ ]:
def sim_weights_batch(X_target_norm, X_ref_norm, K):
    """
    Gaussian similarity weights.
    X_*_norm : already standardised AND divided by σ  →  shape (n, n_features)
    Kernel   : exp(-0.5 * mean_f((x_t_f - x_r_f)²))
    Uses mean (not sum) → O(1) regardless of n_features; prevents underflow.
    """
    diff = X_target_norm[:, None, :] - X_ref_norm[None, :, :]  # (nt, nr, nf)
    S    = np.exp(-0.5 * np.mean(diff**2, axis=2))              # (nt, nr)

    K_int  = int(np.floor(K))
    K_frac = K - K_int
    idx_sorted = np.argsort(-S, axis=1)
    weights    = np.zeros_like(S)
    for i in range(S.shape[0]):
        top = idx_sorted[i, :K_int + 1]
        weights[i, top[:K_int]] = S[i, top[:K_int]]
        if K_frac > 0 and K_int < S.shape[1]:
            weights[i, top[K_int]] = K_frac * S[i, top[K_int]]
    return weights


def sim_cv_r2(X_raw, y, sigma_arr, K, fold_splits_,
              batch_size=SIM_BATCH_SIZE, bin_width=HIST_BIN_WIDTH):
    """
    5-fold CV R² for similarity model.
    StandardScaler fitted on train fold only (no leakage).
    """
    q_edges   = np.arange(0, q_clip_max + bin_width, bin_width)
    q_centres = 0.5 * (q_edges[:-1] + q_edges[1:])
    r2_folds  = []

    for tr, te in fold_splits_:
        sc      = StandardScaler().fit(X_raw[tr])
        X_tr_n  = sc.transform(X_raw[tr]) / sigma_arr
        X_te_n  = sc.transform(X_raw[te]) / sigma_arr

        preds = np.full(len(te), np.nan)
        for start in range(0, len(te), batch_size):
            end = min(start + batch_size, len(te))
            W   = sim_weights_batch(X_te_n[start:end], X_tr_n, K)
            W  /= np.where(W.sum(axis=1, keepdims=True) == 0,
                           1.0, W.sum(axis=1, keepdims=True))
            for i in range(end - start):
                hist, _ = np.histogram(y[tr], bins=q_edges, weights=W[i])
                cdf = np.cumsum(hist)
                if cdf[-1] > 0:
                    cdf  = cdf / cdf[-1]
                    idx_q = np.searchsorted(cdf, 0.50)
                    preds[start + i] = q_centres[min(idx_q, len(q_centres) - 1)]

        valid = np.isfinite(preds)
        if valid.sum() < 10:
            return -1.0
        r2_folds.append(r2_score(y[te][valid], preds[valid]))

    return float(np.mean(r2_folds))


print("Similarity helper functions ready.")


In [ ]:
sigma_start_arr = np.array([SIGMA_START[f] for f in available_sweep])
r2_baseline_sim = sim_cv_r2(X_sim, y_sim, sigma_start_arr, K_FIXED_FOR_SIGMA, fold_splits)
print(f"Baseline CV R² (σ=1.0, K={K_FIXED_FOR_SIGMA}): {r2_baseline_sim:.4f}")
if r2_baseline_sim < -1:
    print("⚠ R² strongly negative — check data / parquet before sweeping.")
elif r2_baseline_sim < 0:
    print("⚠ R² slightly negative — σ=1.0 too tight; Optuna should improve.")
else:
    print("✓ Positive baseline — proceeding to σ optimisation.")


---
## 8. Similarity σ optimisation (Optuna)

Joint optimisation of all feature bandwidths. K fixed at `K_FIXED_FOR_SIGMA`.


In [ ]:
sigma_csv   = sweep_dir / "sigma_results.csv"
_t_sig_start = time.time()

if RUN_SIGMA_SWEEP:
    def objective_sigma(trial):
        s_arr = np.array([
            trial.suggest_float(f, SIGMA_BOUNDS[0], SIGMA_BOUNDS[1], log=True)
            for f in available_sweep
        ])
        return sim_cv_r2(X_sim, y_sim, s_arr, K_FIXED_FOR_SIGMA, fold_splits)

    sampler    = optuna.samplers.TPESampler(seed=random_state)
    study_sigma = optuna.create_study(direction="maximize", sampler=sampler)
    # Warm start: σ=1.0 for all features
    study_sigma.enqueue_trial({f: 1.0 for f in available_sweep})
    study_sigma.optimize(objective_sigma, n_trials=N_OPTUNA_TRIALS,
                         show_progress_bar=True)

    best_sigmas   = {f: study_sigma.best_params[f] for f in available_sweep}
    best_sigma_r2 = study_sigma.best_value

    study_sigma.trials_dataframe().to_csv(sigma_csv, index=False)
    print(f"\nBest CV R² (σ opt, K={K_FIXED_FOR_SIGMA}): {best_sigma_r2:.4f}")
    for f, v in best_sigmas.items():
        print(f"  '{f}': {v:.4f}")

elif sigma_csv.exists():
    df_trials     = pd.read_csv(sigma_csv)
    best_row_sig  = df_trials.loc[df_trials["value"].idxmax()]
    best_sigmas   = {f: float(best_row_sig[f"params_{f}"]) for f in available_sweep}
    best_sigma_r2 = float(best_row_sig["value"])
    print(f"Loaded {sigma_csv}  ({len(df_trials)} trials)")
    print(f"Best CV R² : {best_sigma_r2:.4f}")
else:
    print("No saved results — set RUN_SIGMA_SWEEP = True.")
    best_sigmas   = SIGMA_START.copy()
    best_sigma_r2 = None

best_sigmas_arr = np.array([best_sigmas[f] for f in available_sweep])
_t_sig_end = time.time()


### 8a. σ result figure

In [ ]:
if best_sigma_r2 is not None:
    best_vals = best_sigmas_arr
    y_pos     = np.arange(len(available_sweep))
    colors    = ["#d73027" if v < 1 else "#4575b4" for v in best_vals]

    fig, axes = plt.subplots(1, 2, figsize=(14, max(5, len(available_sweep) * 0.38)))

    # log scale
    axes[0].barh(y_pos, np.log10(best_vals), color=colors, alpha=0.8)
    axes[0].set_yticks(y_pos); axes[0].set_yticklabels(available_sweep, fontsize=8)
    axes[0].set_xlabel("log₁₀(σ_opt)", fontsize=9)
    axes[0].set_title("Optimised σ (blue > 1 std = loose, red < 1 std = tight)", fontsize=10)
    axes[0].axvline(0, color="k", lw=1.0, ls="--", label="σ=1")
    axes[0].legend(fontsize=8); axes[0].grid(True, axis="x", alpha=0.3)

    # linear scale
    axes[1].barh(y_pos, best_vals, color=colors, alpha=0.8)
    axes[1].set_yticks(y_pos); axes[1].set_yticklabels(available_sweep, fontsize=8)
    axes[1].set_xlabel("σ_opt (×std dev)", fontsize=9)
    axes[1].set_title("Optimised σ — linear scale", fontsize=10)
    axes[1].axvline(1.0, color="k", lw=1.0, ls="--", label="σ=1")
    axes[1].legend(fontsize=8); axes[1].grid(True, axis="x", alpha=0.3)

    fig.suptitle(
        f"Similarity σ optimisation | K={K_FIXED_FOR_SIGMA} | "
        f"CV R²={best_sigma_r2:.4f} | {N_OPTUNA_TRIALS} trials",
        fontsize=11, fontweight="bold",
    )
    fig.tight_layout()
    fig.savefig(sweep_dir / "sim_sigma_sweep.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/sim_sigma_sweep.png")


---
## 9. Similarity K sweep

Grid search over K values using optimised σ from §8.


In [ ]:
k_csv       = sweep_dir / "k_results.csv"
_t_k_start  = time.time()

if RUN_K_SWEEP:
    K_values = np.linspace(K_RANGE[0], K_RANGE[1], K_SWEEP_STEPS)
    k_rows   = []
    for K_val in K_values:
        r2 = sim_cv_r2(X_sim, y_sim, best_sigmas_arr, K_val, fold_splits)
        k_rows.append({"K": K_val, "cv_r2": r2})
        print(f"  K={K_val:5.2f}  CV R²={r2:.4f}")

    k_df    = pd.DataFrame(k_rows)
    best_K  = float(k_df.loc[k_df["cv_r2"].idxmax(), "K"])
    best_K_r2 = float(k_df["cv_r2"].max())
    k_df.to_csv(k_csv, index=False)
    print(f"\nBest K = {best_K:.2f}  CV R²={best_K_r2:.4f}")

elif k_csv.exists():
    k_df    = pd.read_csv(k_csv)
    best_K  = float(k_df.loc[k_df["cv_r2"].idxmax(), "K"])
    best_K_r2 = float(k_df["cv_r2"].max())
    print(f"Loaded {k_csv}")
    print(f"Best K = {best_K:.2f}  CV R²={best_K_r2:.4f}")
else:
    k_df    = None
    best_K  = 7.0    # sensible default before sweep
    best_K_r2 = None
    print("No saved results — set RUN_K_SWEEP = True.")

_t_k_end = time.time()


### 9a. K sweep figure

In [ ]:
if k_df is not None:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(k_df["K"], k_df["cv_r2"], "o-", ms=4, color="steelblue", lw=1.2)
    ax.axvline(best_K, color="firebrick", lw=1.5, ls="--",
               label=f"Best K = {best_K:.2f} (R²={best_K_r2:.4f})")
    ax.set_xlabel("K (continuous)", fontsize=10)
    ax.set_ylabel("5-fold CV R²", fontsize=10)
    ax.set_title(
        f"Similarity model — K sweep | {len(available_sweep)} features | "
        f"optimised σ",
        fontsize=11,
    )
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(sweep_dir / "sim_k_sweep.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/sim_k_sweep.png")


### 9b. Summary table (σ baseline → σ-opt → K-opt)

In [ ]:
combos = [
    ("baseline (σ=1, K=7)",        sigma_start_arr, 7.0),
    (f"best σ, K_fixed={K_FIXED_FOR_SIGMA}", best_sigmas_arr, K_FIXED_FOR_SIGMA),
    (f"best σ, best K={best_K:.2f}",          best_sigmas_arr, best_K),
]
summary_rows = []
for label, s_arr, k_val in combos:
    r2 = sim_cv_r2(X_sim, y_sim, s_arr, k_val, fold_splits)
    summary_rows.append({"configuration": label, "CV_R2": round(r2, 4)})
    print(f"  {label:<45s} R²={r2:.4f}")

sim_summary_df = pd.DataFrame(summary_rows)
sim_summary_df.to_csv(sweep_dir / "sim_summary.csv", index=False)
print("\nSaved: output/sweeps/sim_summary.csv")


---
## 10. Similarity feature sensitivity

Three sub-tests:
- **LOO** — ΔR² when each feature is removed from the full model
- **Single-feature** — univariate CV R²
- **σ sensitivity** — R² as σᵢ sweeps from tight→loose (others fixed)


### 10a. Leave-one-out

In [ ]:
loo_csv = sweep_dir / "sim_loo_results.csv"
r2_full = sim_cv_r2(X_sim, y_sim, best_sigmas_arr, best_K, fold_splits)
print(f"Full model CV R² (best σ, K={best_K:.2f}): {r2_full:.4f}")

if RUN_FEATURE_SENS and RUN_LOO:
    loo_rows = []
    n = len(available_sweep)
    for i, feat in enumerate(available_sweep):
        idx_keep = [j for j in range(n) if j != i]
        s_sub    = best_sigmas_arr[idx_keep]
        r2_sub   = sim_cv_r2(X_sim[:, idx_keep], y_sim, s_sub, best_K, fold_splits)
        delta    = r2_sub - r2_full
        tag = "HELPFUL" if delta < -0.002 else ("HARMFUL" if delta > 0.002 else "neutral")
        loo_rows.append({"feature": feat, "r2_without": r2_sub,
                         "delta_r2": delta, "verdict": tag})
        print(f"  [{i+1:2d}/{n}] drop {feat:<22s} R²={r2_sub:.4f} Δ={delta:+.4f} {tag}")

    loo_df = pd.DataFrame(loo_rows).sort_values("delta_r2", ascending=False)
    loo_df.to_csv(loo_csv, index=False)
    print(f"\nSaved {loo_csv}")

elif loo_csv.exists():
    loo_df = pd.read_csv(loo_csv)
    print(f"Loaded {loo_csv}")
else:
    loo_df = None
    print("Set RUN_LOO = True to run.")


### 10b. Single-feature (univariate)

In [ ]:
single_csv = sweep_dir / "sim_single_results.csv"

if RUN_FEATURE_SENS and RUN_SINGLE:
    single_rows = []
    for i, feat in enumerate(available_sweep):
        s_one  = np.array([best_sigmas_arr[i]])
        r2_one = sim_cv_r2(X_sim[:, [i]], y_sim, s_one, best_K, fold_splits)
        single_rows.append({"feature": feat, "r2_univariate": r2_one})
        print(f"  {feat:<25s} R²={r2_one:.4f}")

    single_df = pd.DataFrame(single_rows).sort_values("r2_univariate", ascending=False)
    single_df.to_csv(single_csv, index=False)
    print(f"\nSaved {single_csv}")

elif single_csv.exists():
    single_df = pd.read_csv(single_csv)
    print(f"Loaded {single_csv}")
else:
    single_df = None
    print("Set RUN_SINGLE = True to run.")


### 10c. Per-feature σ sensitivity

In [ ]:
sigma_sens_csv = sweep_dir / "sim_sigma_sensitivity.csv"

if RUN_FEATURE_SENS and RUN_SIGMA_SENS:
    sigma_grid = np.logspace(
        np.log10(SIGMA_BOUNDS[0]), np.log10(SIGMA_BOUNDS[1]), SIGMA_SENS_STEPS
    )
    sens_rows = []
    for i, feat in enumerate(available_sweep):
        feat_best_r2, feat_best_s = -np.inf, None
        for s_val in sigma_grid:
            s_test    = best_sigmas_arr.copy()
            s_test[i] = s_val
            r2_s = sim_cv_r2(X_sim, y_sim, s_test, best_K, fold_splits)
            sens_rows.append({"feature": feat, "sigma": s_val, "cv_r2": r2_s})
            if r2_s > feat_best_r2:
                feat_best_r2, feat_best_s = r2_s, s_val
        print(f"  {feat:<25s} best σ={feat_best_s:.3f} R²={feat_best_r2:.4f} "
              f"(joint best={best_sigmas_arr[i]:.4f})")

    sigma_sens_df = pd.DataFrame(sens_rows)
    sigma_sens_df.to_csv(sigma_sens_csv, index=False)
    print(f"\nSaved {sigma_sens_csv}")

elif sigma_sens_csv.exists():
    sigma_sens_df = pd.read_csv(sigma_sens_csv)
    print(f"Loaded {sigma_sens_csv}")
else:
    sigma_sens_df = None
    print("Set RUN_SIGMA_SENS = True to run.")


### 10d. Combined LOO + univariate figure

In [ ]:
if 'loo_df' in dir() and loo_df is not None and \
   'single_df' in dir() and single_df is not None:
    merged = (loo_df[["feature","delta_r2","verdict"]]
              .merge(single_df[["feature","r2_univariate"]], on="feature")
              .sort_values("delta_r2"))  # most HELPFUL at top

    n_feat = len(merged)
    y_pos  = np.arange(n_feat)
    fig, axes = plt.subplots(1, 2, figsize=(14, max(5, n_feat * 0.38)))

    colors_loo = ["#1a9850" if d < -0.002 else "#d73027" if d > 0.002 else "#999999"
                  for d in merged["delta_r2"]]
    axes[0].barh(y_pos, merged["delta_r2"], color=colors_loo, alpha=0.85)
    axes[0].set_yticks(y_pos); axes[0].set_yticklabels(merged["feature"], fontsize=8)
    axes[0].axvline(0, color="k", lw=0.8)
    axes[0].set_xlabel("ΔR² (drop feature vs full model)", fontsize=9)
    axes[0].set_title("Leave-one-out | green=helpful, red=harmful", fontsize=10)
    axes[0].grid(True, axis="x", alpha=0.3)

    colors_uni = ["#4575b4" if r > 0 else "#d73027" for r in merged["r2_univariate"]]
    axes[1].barh(y_pos, merged["r2_univariate"], color=colors_uni, alpha=0.85)
    axes[1].set_yticks(y_pos); axes[1].set_yticklabels(merged["feature"], fontsize=8)
    axes[1].axvline(0, color="k", lw=0.8)
    axes[1].set_xlabel("Univariate CV R² (feature alone)", fontsize=9)
    axes[1].set_title("Single-feature predictive power", fontsize=10)
    axes[1].grid(True, axis="x", alpha=0.3)

    fig.suptitle(
        f"Similarity feature analysis | full model R²={r2_full:.4f} | K={best_K:.2f}",
        fontsize=11, fontweight="bold",
    )
    fig.tight_layout()
    fig.savefig(sweep_dir / "sim_feature_analysis.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/sim_feature_analysis.png")


### 10e. σ sensitivity heatmap

In [ ]:
if 'sigma_sens_df' in dir() and sigma_sens_df is not None:
    pivot      = sigma_sens_df.pivot_table(index="feature", columns="sigma", values="cv_r2")
    feat_order = pivot.max(axis=1).sort_values(ascending=False).index
    pivot      = pivot.loc[feat_order]

    fig, ax = plt.subplots(figsize=(12, max(5, len(feat_order) * 0.38)))
    im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn",
                   vmin=max(pivot.values.min(), r2_full - 0.05),
                   vmax=r2_full + 0.02)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{v:.2f}" for v in pivot.columns], fontsize=7, rotation=45)
    ax.set_yticks(range(len(feat_order)))
    ax.set_yticklabels(feat_order, fontsize=8)
    ax.set_xlabel("σᵢ (others fixed at best values)", fontsize=9)
    ax.set_title(f"Per-feature σ sensitivity | R²={r2_full:.4f} | K={best_K:.2f}", fontsize=11)

    for row_i, feat in enumerate(feat_order):
        feat_idx  = available_sweep.index(feat)
        best_s    = best_sigmas_arr[feat_idx]
        col_i     = np.argmin(np.abs(pivot.columns - best_s))
        ax.plot(col_i, row_i, "k*", ms=6)

    plt.colorbar(im, ax=ax, label="CV R²", shrink=0.6)
    fig.tight_layout()
    fig.savefig(sweep_dir / "sim_sigma_sensitivity.png", dpi=fig_dpi, bbox_inches="tight")
    plt.show()
    print("Saved: output/sweeps/sim_sigma_sensitivity.png  (★ = Optuna best σ)")


### 10f. Feature ranking table

In [ ]:
if 'loo_df' in dir() and loo_df is not None and \
   'single_df' in dir() and single_df is not None:
    rank_df = (loo_df[["feature","r2_without","delta_r2","verdict"]]
               .merge(single_df[["feature","r2_univariate"]], on="feature")
               .sort_values("delta_r2"))

    if 'sigma_sens_df' in dir() and sigma_sens_df is not None:
        sens_best = (sigma_sens_df.sort_values("cv_r2", ascending=False)
                     .groupby("feature").first()[["sigma"]]
                     .rename(columns={"sigma": "sigma_marginal_best"})
                     .reset_index())
        rank_df = rank_df.merge(sens_best, on="feature", how="left")

    sigma_joint = pd.DataFrame({
        "feature":        available_sweep,
        "sigma_joint_opt": best_sigmas_arr,
    })
    rank_df = rank_df.merge(sigma_joint, on="feature", how="left").round(4)

    rank_csv = sweep_dir / "sim_feature_ranking.csv"
    rank_df.to_csv(rank_csv, index=False)
    print(f"Full model R² = {r2_full:.4f}\n")
    print(rank_df.to_string(index=False))
    print(f"\nSaved {rank_csv}")


---
## 11. Write `sim_best_params.json`

Saves optimal K, all σ values, and full sweep diagnostics.  
This is the single source of truth for `5_Sim.ipynb`.

> **⚠ Note:** `SIM_SIGMAS` and `SIM_BEST_K` are **NOT** in `config.py`.  
> Downstream notebooks must load this JSON.


In [ ]:
sim_json = sweep_dir / "sim_best_params.json"

out = {
    "model"            : "Similarity",
    "sweep_notebook"   : "3b_MODEL_SWEEPS",
    "SIM_BEST_K"       : float(best_K),
    "SIM_SIGMAS"       : {f: float(best_sigmas[f]) for f in available_sweep},
    "kernel"           : "exp(-0.5 * mean_f(((x_t - x_r) / sigma_f)^2))",
    "standardisation"  : "StandardScaler per CV fold (train fold only)",
    "features"         : available_sweep,
    "obs_model"        : obs_model,
    "cv_r2_baseline"   : float(r2_baseline_sim),
    "cv_r2_sigma_opt"  : float(best_sigma_r2) if best_sigma_r2 is not None else None,
    "cv_r2_k_opt"      : float(best_K_r2)     if best_K_r2 is not None else None,
    "K_fixed_sigma"    : K_FIXED_FOR_SIGMA,
    "n_optuna_trials"  : N_OPTUNA_TRIALS,
    "excluded": {
        "categorical"       : ["GLiM", "REG", "TC1"],
        "distance_deferred" : ["VOLC_DIST"],
        "uncertainty_fields": ["MOHO_U", "MOHO_GRAV_U"],
    },
}

with open(sim_json, "w") as fp:
    json.dump(out, fp, indent=2)
print(f"Saved {sim_json}")
print(f"Best K       : {out['SIM_BEST_K']:.2f}")
print(f"CV R² baseline → σ-opt → K-opt : "
      f"{out['cv_r2_baseline']:.4f} → "
      f"{out['cv_r2_sigma_opt']} → "
      f"{out['cv_r2_k_opt']}")


---
## 12. Runtime log

Records wall-clock times per stage plus system info.


In [ ]:
import datetime

def _safe(x):
    try: return float(x)
    except: return None

log_entries = [
    {"stage": "param_sweep",         "wall_s": _safe(locals().get("_t_param_end",0) - locals().get("_t_param_start",0))},
    {"stage": "feature_ablation_loo", "wall_s": _safe(locals().get("_t_abl_end",0) - locals().get("_t_abl_start",0))},
    {"stage": "mc_subset_sweep",      "wall_s": _safe(locals().get("_t_mc_end",0) - locals().get("_t_mc_start",0))},
    {"stage": "greedy_forward",       "wall_s": _safe(locals().get("_t_gr_end",0) - locals().get("_t_gr_start",0))},
    {"stage": "sim_sigma_optuna",     "wall_s": _safe(locals().get("_t_sig_end",0) - locals().get("_t_sig_start",0))},
    {"stage": "sim_k_sweep",          "wall_s": _safe(locals().get("_t_k_end",0) - locals().get("_t_k_start",0))},
]

# System info
mem = psutil.virtual_memory()
sys_info = {
    "timestamp"    : datetime.datetime.now().isoformat(),
    "hostname"     : platform.node(),
    "os"           : platform.platform(),
    "python"       : platform.python_version(),
    "cpu_model"    : platform.processor(),
    "cpu_physical" : psutil.cpu_count(logical=False),
    "cpu_logical"  : psutil.cpu_count(logical=True),
    "ram_gb_total" : round(mem.total / 1e9, 1),
    "ram_gb_avail" : round(mem.available / 1e9, 1),
    "param_n_runs" : PARAM_N_RUNS,
    "mc_n_runs"    : MC_N_RUNS,
    "n_optuna_trials": N_OPTUNA_TRIALS,
}

# Try to get GPU info
try:
    import subprocess
    gpu_out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        stderr=subprocess.DEVNULL, text=True,
    ).strip()
    sys_info["gpu"] = gpu_out
except Exception:
    sys_info["gpu"] = "none / not detected"

log_df = pd.DataFrame(log_entries)
log_df["wall_min"] = (log_df["wall_s"] / 60).round(2)

log_out = output_root / "sweep_runtime_log.csv"
log_df.to_csv(log_out, index=False)

# Write sys info as JSON alongside
sys_json = output_root / "sweep_sysinfo.json"
with open(sys_json, "w") as fp:
    json.dump(sys_info, fp, indent=2)

print("=" * 60)
print("Runtime log")
print(log_df[["stage","wall_min"]].to_string(index=False))
print()
print("System info:")
for k, v in sys_info.items():
    print(f"  {k:<20s}: {v}")
print(f"\nSaved {log_out}")
print(f"Saved {sys_json}")
